# 第 3 周练习：合成数据生成器（Synthetic Data Generator）

用 **OpenRouter** 与 **Groq** 两套 OpenAI 兼容 API，并行生成 CSV 格式合成数据，并在 **Gradio** 里左右对比结果。

## 练习目标

1. 从环境变量读取两个 API Key，分别初始化两个 `OpenAI` 客户端（不同 `base_url`）。
2. 用同一套 system/user prompt 调两个模型，要求「只输出 CSV」。
3. 把模型文本解析成 `pandas.DataFrame`，在界面并排展示。

## 怎么跑

1. 准备 `.env`：`OPENROUTER_API_KEY`、`GROQ_API_KEY`。
2. 从上到下运行单元格，最后 `demo.launch` 会打开浏览器界面。
3. 在文本框描述数据集（行数、字段等），点 Generate，对比两侧表格。


In [ ]:
# ========== 导入：API 客户端、Gradio、表格解析 ==========

# os：读环境变量里的 API Key
import os
# OpenAI 官方 SDK：也兼容 OpenRouter / Groq 的 OpenAI-style 接口
from openai import OpenAI
# Gradio：搭并排对比的 Web UI
import gradio as gr
# load_dotenv：把 .env 载入环境变量，避免密钥写进笔记本
from dotenv import load_dotenv
# pandas：把 CSV 文本变成 DataFrame 展示
import pandas as pd
# io.StringIO：把字符串当成「文件」给 pd.read_csv 读
import io


In [ ]:
# ========== 加载环境变量并检查两个 Key 是否存在 ==========

# override=True：.env 里的值覆盖进程里已有同名环境变量
load_dotenv(override=True)
# 从环境读取 OpenRouter 密钥
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
# 从环境读取 Groq 密钥
groq_api_key = os.getenv("GROQ_API_KEY")

# 友好提示：有没有设 Key（打印文案保持原样，便于对照日志）
if openrouter_api_key:
    print("OPENROUTER_API_KEY is set.")
else:
    print("OPENROUTER_API_KEY is not set.")

if groq_api_key:
    print("GROQ_API_KEY is set.")
else:
    print("GROQ_API_KEY is not set.")


In [ ]:
# ========== 常量：模型 id 与 API base URL（字符串勿改） ==========

# OpenRouter 上的 GPT 小模型路由名
MODEL_GPT = 'openai/gpt-4o-mini'
# Groq 上的 Llama 模型名
MODEL_GROQ = 'llama-3.3-70b-versatile'
# Groq 的 OpenAI 兼容端点
GROQ_URL = "https://api.groq.com/openai/v1"
# OpenRouter 的 OpenAI 兼容端点
OPENROUTER_URL = "https://openrouter.ai/api/v1"


In [ ]:
# ========== 初始化两个 OpenAI 兼容客户端 ==========

# OpenRouter：api_key + 自定义 base_url
openrouter = OpenAI(api_key=openrouter_api_key, base_url=OPENROUTER_URL)
# Groq：同样用 OpenAI SDK，只换密钥与 URL
groq_client = OpenAI(api_key=groq_api_key, base_url=GROQ_URL)


In [ ]:
# ========== system prompt：规定模型只输出 CSV（英文指令保持原样） ==========

system_prompt = """ 
You are a synthetic data generator.
Your task is to take a description of a dataset and generate synthetic data that matches the description.
Respond with only the generated data in csv format, without any additional text or explanations.
"""


In [ ]:
# ========== 组装 user prompt：把用户描述嵌进固定模板 ==========

def user_prompt(description):
    # 返回发给模型的 user 内容；模板英文不翻译，以免改变输出格式约束
    return f"""Generate synthetic data based on the following description:
{description}
Respond with only the generated data in csv format, without any additional text or explanations.
"""


In [ ]:
# ========== 把模型返回的 CSV 文本解析成 DataFrame ==========

def process_response(response):
    try:
        # StringIO：让 pandas 像读文件一样读字符串
        df = pd.read_csv(io.StringIO(response))
        return df
    except Exception as e:
        # 解析失败（模型加了说明文字、格式坏了）时打印错误并返回 None
        print(f"Error processing response: {e}")
        return None


In [ ]:
# ========== 主流程：同一描述并行调 Groq 与 OpenRouter ==========

def generate_synthetic_data(description):
    # 空描述直接拒绝，避免白白打 API
    if not description:
        print("Description is empty. Please provide a valid description.")
        return None, None
    # 拼出 user 侧提示词
    prompt = user_prompt(description)
    # 先问 Groq（Llama）
    response_groq = groq_client.chat.completions.create(
        model=MODEL_GROQ,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
    )
    # 再问 OpenRouter（GPT）
    response_openrouter = openrouter.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
    )
    # 取 choices[0].message.content 文本，再转成表
    df_groq = process_response(response_groq.choices[0].message.content)
    df_openrouter = process_response(response_openrouter.choices[0].message.content)
    # 注意：返回顺序是 (groq, openrouter)，与 Gradio outputs 绑定一致
    return df_groq, df_openrouter


In [ ]:
# ========== Gradio：左侧描述 + 右侧两个模型的 Dataframe ==========

with gr.Blocks() as demo:
    gr.Markdown("### Synthetic Data Generator")
    with gr.Row():
        with gr.Column():
            # 用户用自然语言描述想要的数据集
            description_input = gr.Textbox(label="Dataset Description", placeholder="e.g., A dataset of 10 customers with columns for name, age, and city.")
            generate_button = gr.Button("Generate Data", variant="primary")
    with gr.Row():
        with gr.Column():
            # 标签里嵌入模型名常量，方便对照是哪一侧
            df_openai = gr.Dataframe(label=f"{MODEL_GPT} Generated Data")
        with gr.Column():
            df_groq = gr.Dataframe(label=f"{MODEL_GROQ} Generated Data")
    # outputs 顺序：[df_groq, df_openai] —— 对应函数返回的 (df_groq, df_openrouter)
    generate_button.click(
        fn=generate_synthetic_data,
        inputs=description_input,
        outputs=[df_groq, df_openai]
    )

# inbrowser=True：本地自动打开浏览器
demo.launch(inbrowser=True)
